# 10 · Website data export

Writes `website/src/data/researchData.ts` from the saved results, so the research website shows exactly the numbers produced by notebooks 02–09. Run it after notebooks 08 and 09; nothing here trains a model or touches `KDDTest+`.

**Output:** `website/src/data/researchData.ts` (generated; do not edit by hand).

In [1]:
import sys, json, re
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from src import controlled_framework as cf

IDS = ["baseline", "lr", "dt", "rf", "xgb"]
LEARNED = ["lr", "dt", "rf", "xgb"]
KEY = {"baseline": "majority_baseline", "lr": "logistic_regression", "dt": "decision_tree", "rf": "random_forest", "xgb": "xgboost"}
NAME = {"baseline": "Majority Baseline", "lr": "Logistic Regression", "dt": "Decision Tree", "rf": "Random Forest", "xgb": "XGBoost"}
ID_OF_KEY = {v: k for k, v in KEY.items()}
ID_OF_NAME = {v: k for k, v in NAME.items()}

M = {i: json.loads((cf.METRICS_DIR / f"{KEY[i]}.json").read_text()) for i in IDS}
STAT = json.loads((cf.METRICS_DIR / "statistical_comparison.json").read_text())
SEEDS = pd.read_csv(cf.TABLES_DIR / "all_models_seeds.csv")
TESTS = pd.read_csv(cf.TABLES_DIR / "hypothesis_tests.csv")
SEED_PAIRS = pd.read_csv(cf.TABLES_DIR / "seed_paired_differences.csv")
SWEEP = pd.read_csv(cf.TABLES_DIR / "capacity_sweep.csv")
SWEEP_SUM = pd.read_csv(cf.TABLES_DIR / "capacity_sweep_summary.csv")
pct = lambda v, d=2: round(float(v) * 100, d)
r4 = lambda v: round(float(v), 4)
BUDGETS = [cf.budget_key(b) for b in cf.FPR_BUDGETS]
print("loaded results for", IDS)

loaded results for ['baseline', 'lr', 'dt', 'rf', 'xgb']


## 1. Models, metrics and operating points

In [2]:
TEXT = {
    "baseline": ("Heuristic floor", "Zero-rule floor: always predicts the training majority class (normal).",
                 "Gives 43.07% accuracy on the inverted test distribution with zero attack recall; its score is constant, so its ROC-AUC is 0.5."),
    "lr": ("Regularized linear", "Single linear model on the shared 122-feature input.",
           "L2-regularised logistic regression. The novelty-aware objective selected strong regularisation."),
    "dt": ("Single tree", "Single CART tree: the interpretable non-linear model.",
           "Depth, split and leaf sizes, criterion and feature subsampling are tuned; the novelty-aware objective selected a shallow tree."),
    "rf": ("Bagged ensemble", "High-capacity bagged ensemble of CART trees with random feature subspaces.",
           "Number of trees, depth, split and leaf sizes and max_features are tuned."),
    "xgb": ("Boosted ensemble", "High-capacity gradient-boosted tree ensemble (XGBoost, histogram method).",
            "Number of trees, depth, learning rate, child weight, subsampling and L2 strength are tuned; scale_pos_weight=1 is the equivalent of class_weight=None."),
}
DROP = {"n_jobs", "eval_metric", "objective", "tree_method", "best_novelty_cv_auc", "best_novelty_cv_auc_std", "bootstrap", "solver", "max_iter"}
def hp(i):
    out = {}
    for k, v in M[i]["hyperparameters"].items():
        if k in DROP: continue
        out[k] = round(v, 4) if isinstance(v, float) else v
    return out

models = []
for i in IDS:
    h = M[i]["hyperparameters"]
    models.append({"id": i, "name": NAME[i] if i != "baseline" else "Majority-Class Baseline",
                   "shortName": NAME[i] if i != "baseline" else "Baseline",
                   "type": TEXT[i][0], "role": TEXT[i][1], "description": TEXT[i][2],
                   "cvAuc": r4(h["best_novelty_cv_auc"]) if "best_novelty_cv_auc" in h else None,
                   "cvAucStd": r4(h["best_novelty_cv_auc_std"]) if "best_novelty_cv_auc_std" in h else None,
                   "latencyUs": M[i]["timing"]["inference_us_per_sample"],
                   "hyperparameters": hp(i),
                   "complexity": {k: round(float(v), 2) for k, v in M[i].get("model_complexity", {}).items()}})

boot = STAT["bootstrap_95_confidence_intervals"]
def row(metric, key, unit, desc, hib, getter, fmt):
    vals = {i: getter(i) for i in IDS}
    return {"metric": metric, "key": key, "unit": unit, "description": desc, "higherIsBetter": hib,
            "values": vals, "display": {i: fmt(i, vals[i]) for i in IDS}}
o = lambda i, k: M[i]["overall_metrics"][k]
tf = lambda i, k: M[i]["threshold_free"][k]
cm = lambda i: M[i]["overall_metrics"]["confusion_matrix"]
def f1_ci(i, v):
    k = KEY[i]
    return f"{v:.4f}" if k not in boot else f"{v:.4f} [{boot[k]['macro_f1']['ci_95_lower']:.3f}, {boot[k]['macro_f1']['ci_95_upper']:.3f}]"

metrics = [
    row("ROC-AUC", "roc_auc", "score", "Threshold-free ranking quality over all test rows.", True, lambda i: r4(tf(i, "roc_auc")), lambda i, v: f"{v:.4f}"),
    row("PR-AUC", "pr_auc", "score", "Average precision for the attack class (threshold-free).", True, lambda i: r4(tf(i, "pr_auc")), lambda i, v: f"{v:.4f}"),
    row("AUC: unseen attacks vs normal", "auc_unseen", "score", "Ranking of the 3,750 unseen-attack rows against the 9,710 normal rows.", True, lambda i: r4(tf(i, "roc_auc_unseen_vs_normal")), lambda i, v: f"{v:.4f}"),
    row("AUC: seen attacks vs normal", "auc_seen", "score", "Ranking of the 9,083 seen-attack rows against the 9,710 normal rows.", True, lambda i: r4(tf(i, "roc_auc_seen_vs_normal")), lambda i, v: f"{v:.4f}"),
    row("Accuracy @0.5", "accuracy", "%", "Share of all 22,543 test rows classified correctly at threshold 0.5.", True, lambda i: pct(o(i, "accuracy")), lambda i, v: f"{v:.2f}%"),
    row("Macro-F1 @0.5 (95% CI)", "macro_f1", "score", "Unweighted mean F1 over normal and attack at threshold 0.5; row-bootstrap CI.", True, lambda i: r4(o(i, "macro_f1")), f1_ci),
    row("Attack recall @0.5", "attack_recall", "%", "Share of the 12,833 attack rows flagged at threshold 0.5.", True, lambda i: pct(o(i, "attack_recall")), lambda i, v: f"{v:.2f}%"),
    row("Attack precision @0.5", "attack_precision", "%", "Share of alarms that are real attacks at threshold 0.5.", True, lambda i: pct(o(i, "attack_precision")), lambda i, v: f"{v:.2f}%"),
    row("False-positive rate @0.5", "fpr", "%", "Share of the 9,710 normal rows flagged as attacks at threshold 0.5.", False, lambda i: pct(o(i, "false_positive_rate")), lambda i, v: f"{v:.2f}% ({cm(i)[0][1]:,})"),
]

confusion = {}
for i in IDS:
    (tn, fp), (fn, tp) = cm(i)
    confusion[i] = {"tn": tn, "fp": fp, "fn": fn, "tp": tp, "total": tn + fp + fn + tp,
                    "precision": pct(o(i, "attack_precision")), "recall": pct(o(i, "attack_recall")),
                    "fpr": pct(o(i, "false_positive_rate")), "fnr": pct(o(i, "false_negative_rate")),
                    "specificity": pct(o(i, "specificity"))}

threshold_free = {}
for i in IDS:
    ops = M[i]["test_roc_operating_points"]
    threshold_free[i] = {"rocAuc": r4(tf(i, "roc_auc")), "prAuc": r4(tf(i, "pr_auc")),
                         "aucSeen": r4(tf(i, "roc_auc_seen_vs_normal")), "aucUnseen": r4(tf(i, "roc_auc_unseen_vs_normal")),
                         "unseenAtFpr": {b: pct(ops[b]["unseen_recall"]) for b in BUDGETS},
                         "seenAtFpr": {b: pct(ops[b]["seen_recall"]) for b in BUDGETS},
                         "achievedFpr": {b: pct(ops[b]["fpr"]) for b in BUDGETS}}
deployable = {i: [{"budget": b, "fpr": pct(v["fpr"]), "attackRecall": pct(v["attack_recall"]),
                   "seenRecall": pct(v["seen_recall"]), "unseenRecall": pct(v["unseen_recall"])}
                  for b, v in M[i]["deployable_operating_points"].items()] for i in LEARNED}
print(len(metrics), "metric rows")

9 metric rows


## 2. Seeds, categories, generalisation, unseen attacks

In [3]:
SEED_COLS = {"cvAuc": "best_novelty_cv_auc", "rocAuc": "roc_auc", "aucSeen": "auc_seen_vs_normal",
             "aucUnseen": "auc_unseen_vs_normal", "unseenAt2": "unseen_recall@test_fpr_2pct",
             "macroF1": "macro_f1", "fpr": "false_positive_rate", "unseenRecall": "unseen_recall"}
PCT_COLS = {"unseenAt2", "fpr", "unseenRecall"}
seeds = {}
for i in LEARNED:
    s = SEEDS[SEEDS.model == NAME[i]].sort_values("seed")
    seeds[i] = {k: {"mean": (pct if k in PCT_COLS else r4)(s[c].mean()), "std": (pct if k in PCT_COLS else r4)(s[c].std()),
                    "values": [(pct if k in PCT_COLS else r4)(v) for v in s[c]]} for k, c in SEED_COLS.items()}
seed_list = [int(v) for v in sorted(SEEDS.seed.unique())]

LABEL = {"DoS": "DoS (Denial of Service)", "Probe": "Probe (Surveillance)", "R2L": "R2L (Remote to Local)", "U2R": "U2R (User to Root)"}
categories = []
for c in cf.ATTACK_CATEGORIES:
    st = M["lr"]["category_stratified_seen_unseen_recall"][c]
    categories.append({"category": c, "label": LABEL[c], "total": M["lr"]["attack_category_recall"][c]["total_samples"],
                       "recall": {i: pct(M[i]["attack_category_recall"][c]["recall"]) for i in IDS},
                       "detected": {i: M[i]["attack_category_recall"][c]["detected"] for i in IDS},
                       "seenN": st["seen"]["total_samples"], "unseenN": st["unseen"]["total_samples"],
                       "seenRecall": {i: pct(M[i]["category_stratified_seen_unseen_recall"][c]["seen"]["recall"]) for i in LEARNED},
                       "unseenRecall": {i: pct(M[i]["category_stratified_seen_unseen_recall"][c]["unseen"]["recall"]) for i in LEARNED}})

cl = STAT["cluster_bootstrap_unseen_recall"]
sv = lambda i, k: M[i]["seen_vs_unseen_recall"][k]
generalization = {
    "totalSeen": sv("lr", "seen_total"), "totalUnseen": sv("lr", "unseen_total"),
    "seenRecall": {i: pct(sv(i, "seen_recall")) for i in IDS},
    "unseenRecall": {i: pct(sv(i, "unseen_recall")) for i in IDS},
    "dropPp": {i: sv(i, "generalization_drop_percentage_points") for i in IDS},
    "reweightedDropPp": {i: sv(i, "reweighted_drop_percentage_points") for i in LEARNED},
    "macroUnseenRecall": {i: pct(sv(i, "unseen_macro_recall_over_types")) for i in IDS},
    "seenRowCI": {i: [pct(boot[KEY[i]]["seen_recall"]["ci_95_lower"]), pct(boot[KEY[i]]["seen_recall"]["ci_95_upper"])] for i in LEARNED},
    "unseenRowCI": {i: [pct(boot[KEY[i]]["unseen_recall"]["ci_95_lower"]), pct(boot[KEY[i]]["unseen_recall"]["ci_95_upper"])] for i in LEARNED},
    "unseenClusterCI": {i: [pct(cl[KEY[i]]["unseen_recall_micro"]["ci_95_lower"]), pct(cl[KEY[i]]["unseen_recall_micro"]["ci_95_upper"])] for i in LEARNED},
    "aucSeen": {i: r4(tf(i, "roc_auc_seen_vs_normal")) for i in LEARNED},
    "aucUnseen": {i: r4(tf(i, "roc_auc_unseen_vs_normal")) for i in LEARNED},
    "unseenCategoryMix": {c: round(M["lr"]["category_stratified_seen_unseen_recall"][c]["unseen"]["total_samples"] / sv("lr", "unseen_total") * 100, 1)
                          for c in cf.ATTACK_CATEGORIES},
}

unseen_attacks = []
for a, v in M["lr"]["per_unseen_attack_recall"].items():
    unseen_attacks.append({"name": a, "category": v["category"], "count": v["total_samples"],
                           "results": {i: {"detected": M[i]["per_unseen_attack_recall"][a]["detected"],
                                           "missed": M[i]["per_unseen_attack_recall"][a]["missed"],
                                           "recall": pct(M[i]["per_unseen_attack_recall"][a]["recall"])} for i in IDS}})
unseen_attacks.sort(key=lambda d: -d["count"])
print(len(categories), "categories |", len(unseen_attacks), "unseen attacks | seeds", seed_list)

4 categories | 17 unseen attacks | seeds [1, 2, 3, 4, 42]


## 3. Statistical tests and capacity sweeps

In [4]:
FAMILY = {"mcnemar_full": ("McNemar · all test rows", "0-1 error"), "mcnemar_unseen": ("McNemar · unseen-attack rows", "detection")}
METRIC = {"macro_f1": "Δ Macro-F1 @0.5", "unseen_recall_pp": "Δ unseen recall @0.5 (pp)",
          "auc_unseen_vs_normal": "Δ AUC unseen vs normal", "unseen_recall_at_test_fpr_2pct_pp": "Δ unseen recall @2% FPR (pp)"}
tests = []
for t in TESTS.itertuples(index=False):
    a_key, b_key = t.pair.split("_vs_")
    a, b = ID_OF_KEY[a_key], ID_OF_KEY[b_key]
    if t.test.startswith("mcnemar"):
        fam, met = FAMILY[t.test]
        ct = STAT["mcnemar_tests_full_test_set" if t.test == "mcnemar_full" else "mcnemar_tests_unseen_attacks"][t.pair]["contingency_table"]
        extra = {"aBetter": ct["m1_correct_m2_wrong"], "bBetter": ct["m1_wrong_m2_correct"]}
        ci = None
    else:
        fam, met = "Paired bootstrap", METRIC[t.test.split(":", 1)[1]]
        lo, hi = [float(x) for x in str(t.ci_95).strip("[]").split(",")]
        ci, extra = [round(lo, 4), round(hi, 4)], {}
    tests.append({"family": fam, "metric": met, "a": a, "b": b, "effect": round(float(t.effect), 4), "ci": ci,
                  "pRaw": float(f"{t.p_raw:.3g}"), "pHolm": float(f"{t.p_holm:.3g}"), "significant": bool(t.significant_holm_05), **extra})

seed_pairs = []
for t in SEED_PAIRS.itertuples(index=False):
    a_key, b_key = t.pair.split("_vs_")
    scale = 100 if t.metric != "auc_unseen_vs_normal" else 1
    seed_pairs.append({"a": ID_OF_KEY[a_key], "b": ID_OF_KEY[b_key], "metric": t.metric,
                       "meanDiff": round(t.mean_diff * scale, 4), "sdDiff": round(t.sd_diff * scale, 4),
                       "aWins": int(t.a_wins), "nSeeds": int(t.n_seeds)})

def fmt_value(v):
    """Sweep values come back from CSV as floats/NaN: render 'None', '6', '0.001'."""
    try:
        f = float(v)
    except (TypeError, ValueError):
        return str(v)
    if f != f:
        return "None"
    return str(int(f)) if f.is_integer() else f"{f:g}"

sweeps = []
for name, g in SWEEP.groupby("model", sort=False):
    s = SWEEP_SUM[SWEEP_SUM.model == name].iloc[0]
    sweeps.append({"model": ID_OF_NAME[name], "param": g.param.iloc[0], "tuned": fmt_value(g.tuned_value.iloc[0]),
                   "spearman": round(float(s.spearman_cv_vs_test), 3),
                   "points": [{"value": fmt_value(p.value), "complexity": round(float(p.complexity), 2), "cvAuc": r4(p.novelty_cv_auc),
                               "testAucUnseen": r4(p.test_auc_unseen_vs_normal), "testAucSeen": r4(p.test_auc_seen_vs_normal)}
                              for p in g.itertuples(index=False)]})
n_sig = sum(t["significant"] for t in tests)
print(len(tests), "tests,", n_sig, "significant after Holm |", len(sweeps), "sweeps")

36 tests, 29 significant after Holm | 4 sweeps


## 3b. ROC curves (seed 42) for the interactive ROC explorer

In [5]:
# Recall at "FPR <= x" on a fixed FPR grid, using the same threshold rule as the operating points
# (smallest threshold whose false-positive rate on test normals does not exceed x).
FPR_GRID = np.unique(np.round(np.r_[np.linspace(0, 0.2, 81), np.linspace(0.2, 1, 33)], 4))
roc = {}
for i in LEARNED:
    p = pd.read_csv(cf.PRED_DIR / f"{KEY[i]}_predictions.csv")
    y, s, u = p.y_true.to_numpy(), p.y_prob.to_numpy(), p.is_unseen.to_numpy()
    normals = s[y == 0]
    thresholds = [cf.threshold_for_fpr(normals, x) for x in FPR_GRID]
    groups = {"all": y == 1, "seen": (y == 1) & ~u, "unseen": (y == 1) & u}
    roc[i] = {k: [pct((s[m] > t).mean(), 2) for t in thresholds] for k, m in groups.items()}
roc_grid = [pct(x, 2) for x in FPR_GRID]
print(len(roc_grid), "grid points; unseen recall at 2% FPR:", {i: roc[i]["unseen"][roc_grid.index(2.0)] for i in LEARNED})

113 grid points; unseen recall at 2% FPR: {'lr': 5.39, 'dt': 14.59, 'rf': 9.31, 'xgb': 7.95}


## 4. Dataset facts and protocol

In [6]:
data = cf.load_dataset()
tr, te = data.train_df, data.test_df
n_seen, n_unseen = sv("lr", "seen_total"), sv("lr", "unseen_total")
dataset = {"name": "NSL-KDD Benchmark", "citation": "Tavallaee et al. (CISDA 2009)",
           "trainRows": len(tr), "testRows": len(te), "rawFeatures": 41, "encodedFeatures": 122,
           "trainNormal": int((tr.is_intrusion == 0).sum()), "trainNormalPct": pct((tr.is_intrusion == 0).mean()),
           "trainAttack": int((tr.is_intrusion == 1).sum()), "trainAttackPct": pct((tr.is_intrusion == 1).mean()),
           "testNormal": int((te.is_intrusion == 0).sum()), "testNormalPct": pct((te.is_intrusion == 0).mean()),
           "testAttack": int((te.is_intrusion == 1).sum()), "testAttackPct": pct((te.is_intrusion == 1).mean()),
           "seenAttacksInTest": n_seen, "seenAttacksInTestPct": round(n_seen / (n_seen + n_unseen) * 100, 2),
           "unseenAttacksInTest": n_unseen, "unseenAttacksInTestPct": round(n_unseen / (n_seen + n_unseen) * 100, 2),
           "totalUnseenTypes": len(data.unseen_attacks), "trainAttackTypes": int(tr.loc[tr.is_intrusion == 1, "label"].nunique()),
           "seenTypesInTest": len(data.seen_attacks),
           "featureGroups": [
               {"name": "Basic Features", "count": 10, "description": "Connection-level fields: duration, protocol_type, service, flag, src_bytes, dst_bytes, land, wrong_fragment, urgent, hot."},
               {"name": "Content Features", "count": 12, "description": "Payload-derived indicators: num_failed_logins, logged_in, num_compromised, root_shell, su_attempted, num_root, etc."},
               {"name": "Time-Traffic Features", "count": 9, "description": "Rates over the past 2 seconds: count, srv_count, serror_rate, srv_serror_rate, rerror_rate, same_srv_rate, diff_srv_rate, etc."},
               {"name": "Host-Traffic Features", "count": 10, "description": "Rates over the last 100 connections to the same host: dst_host_count, dst_host_srv_count, dst_host_same_srv_rate, etc."}]}
folds = json.loads((cf.SPLITS_DIR / "attack_type_folds.json").read_text())["attack_types_per_fold"]
protocol = {"nTrials": cf.N_TRIALS, "nSplits": cf.N_SPLITS, "seeds": seed_list, "primarySeed": cf.SEED,
            "threshold": cf.THRESHOLD, "fprBudgets": [pct(b, 0) for b in cf.FPR_BUDGETS], "matchedFpr": pct(cf.MATCHED_FPR, 0),
            "bootstraps": cf.N_BOOTSTRAPS, "nTests": len(tests), "nSignificant": n_sig, "alpha": cf.ALPHA,
            "attackTypeFolds": {str(k): v for k, v in folds.items()}}
print(protocol)

{'nTrials': 25, 'nSplits': 5, 'seeds': [1, 2, 3, 4, 42], 'primarySeed': 42, 'threshold': 0.5, 'fprBudgets': [1.0, 2.0, 5.0], 'matchedFpr': 2.0, 'bootstraps': 2000, 'nTests': 36, 'nSignificant': 29, 'alpha': 0.05, 'attackTypeFolds': {'0': ['guess_passwd', 'neptune', 'nmap', 'perl', 'rootkit'], '1': ['back', 'buffer_overflow', 'loadmodule', 'satan', 'spy'], '2': ['ftp_write', 'ipsweep', 'teardrop', 'warezmaster'], '3': ['land', 'multihop', 'portsweep', 'warezclient'], '4': ['imap', 'phf', 'pod', 'smurf']}}


## 5. Write `website/src/data/researchData.ts`

In [7]:
TS_TYPES = r"""// AUTO-GENERATED by notebooks/10_website_data.ipynb from results/. Do not edit by hand:
// rerun the notebook after any change to the experiment.

export type ModelId = 'baseline' | 'lr' | 'dt' | 'rf' | 'xgb';
export type LearnedId = Exclude<ModelId, 'baseline'>;

export interface ModelInfo {
  id: ModelId;
  name: string;
  shortName: string;
  type: string;
  role: string;
  description: string;
  cvAuc: number | null;
  cvAucStd: number | null;
  latencyUs: number;
  hyperparameters: Record<string, string | number | boolean | null>;
  complexity: Record<string, number>;
}

export interface MetricRow {
  metric: string;
  key: string;
  unit: '%' | 'score';
  description: string;
  higherIsBetter: boolean;
  values: Record<ModelId, number>;
  display: Record<ModelId, string>;
}

export interface ConfusionMatrix {
  tn: number; fp: number; fn: number; tp: number; total: number;
  precision: number; recall: number; fpr: number; fnr: number; specificity: number;
}

export interface ThresholdFree {
  rocAuc: number;
  prAuc: number;
  aucSeen: number;
  aucUnseen: number;
  unseenAtFpr: Record<string, number>;
  seenAtFpr: Record<string, number>;
  achievedFpr: Record<string, number>;
}

export interface OperatingPoint {
  budget: string;
  fpr: number;
  attackRecall: number;
  seenRecall: number;
  unseenRecall: number;
}

export interface SeedStat { mean: number; std: number; values: number[] }

export interface CategoryRow {
  category: 'DoS' | 'Probe' | 'R2L' | 'U2R';
  label: string;
  total: number;
  recall: Record<ModelId, number>;
  detected: Record<ModelId, number>;
  seenN: number;
  unseenN: number;
  seenRecall: Record<LearnedId, number>;
  unseenRecall: Record<LearnedId, number>;
}

export interface UnseenAttackRecord {
  name: string;
  category: 'DoS' | 'Probe' | 'R2L' | 'U2R';
  count: number;
  results: Record<ModelId, { detected: number; missed: number; recall: number }>;
}

export interface TestRow {
  family: string;
  metric: string;
  a: LearnedId;
  b: LearnedId;
  effect: number;
  ci: [number, number] | null;
  pRaw: number;
  pHolm: number;
  significant: boolean;
  aBetter?: number;
  bBetter?: number;
}

export interface SeedPair { a: LearnedId; b: LearnedId; metric: string; meanDiff: number; sdDiff: number; aWins: number; nSeeds: number }

export type RocKind = 'all' | 'seen' | 'unseen';

export interface DatasetSpec {
  name: string; citation: string; trainRows: number; testRows: number; rawFeatures: number; encodedFeatures: number;
  trainNormal: number; trainNormalPct: number; trainAttack: number; trainAttackPct: number;
  testNormal: number; testNormalPct: number; testAttack: number; testAttackPct: number;
  seenAttacksInTest: number; seenAttacksInTestPct: number; unseenAttacksInTest: number; unseenAttacksInTestPct: number;
  totalUnseenTypes: number; trainAttackTypes: number; seenTypesInTest: number;
  featureGroups: { name: string; count: number; description: string }[];
}

export interface SweepSeries {
  model: LearnedId;
  param: string;
  tuned: string;
  spearman: number;
  points: { value: string; complexity: number; cvAuc: number; testAucUnseen: number; testAucSeen: number }[];
}
"""

def const(name, typ, value):
    return f"export const {name}: {typ} = {json.dumps(value, indent=2, ensure_ascii=False)};\n"

parts = [TS_TYPES,
         "export const MODEL_IDS: ModelId[] = ['baseline', 'lr', 'dt', 'rf', 'xgb'];\n",
         "export const LEARNED_IDS: LearnedId[] = ['lr', 'dt', 'rf', 'xgb'];\n",
         const("MODEL_NAMES", "Record<ModelId, string>", NAME),
         const("MODELS", "ModelInfo[]", models),
         const("OVERALL_METRICS", "MetricRow[]", metrics),
         const("CONFUSION_MATRICES", "Record<ModelId, ConfusionMatrix>", confusion),
         const("THRESHOLD_FREE", "Record<ModelId, ThresholdFree>", threshold_free),
         const("DEPLOYABLE", "Record<LearnedId, OperatingPoint[]>", deployable),
         const("SEED_SUMMARY", "Record<LearnedId, Record<string, SeedStat>>", seeds),
         const("SEED_PAIRS", "SeedPair[]", seed_pairs),
         const("ATTACK_CATEGORIES", "CategoryRow[]", categories),
         const("GENERALIZATION_DATA", "any", generalization),
         const("UNSEEN_ATTACKS", "UnseenAttackRecord[]", unseen_attacks),
         const("STATISTICAL_TESTS", "TestRow[]", tests),
         const("CAPACITY_SWEEPS", "SweepSeries[]", sweeps),
         const("ROC_FPR_GRID", "number[]", roc_grid),
         const("ROC_CURVES", "Record<LearnedId, Record<RocKind, number[]>>", roc),
         const("DATASET_SPECIFICATION", "DatasetSpec", dataset),
         const("PROTOCOL", "any", protocol)]
out = PROJECT_ROOT / "website" / "src" / "data" / "researchData.ts"
out.write_text("\n".join(parts))
print("wrote", out.relative_to(PROJECT_ROOT), f"({out.stat().st_size:,} bytes)")

wrote website/src/data/researchData.ts (73,650 bytes)
